In [15]:
import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy import stats

In [3]:
plt.style.use('default')
mpl.rcParams['axes.linewidth'] = 5  # set the value globally
mpl.rcParams['xtick.major.size'] = 20
mpl.rcParams['xtick.major.width'] = 7
mpl.rcParams['xtick.minor.size'] = 10
mpl.rcParams['xtick.minor.width'] = 7
mpl.rcParams['ytick.major.size'] = 20
mpl.rcParams['ytick.major.width'] = 7
mpl.rcParams['ytick.labelsize'] = 50
mpl.rcParams['xtick.labelsize'] = 50
mpl.rcParams['ytick.minor.size'] = 10
mpl.rcParams['ytick.minor.width'] = 7
mpl.rcParams['font.size'] = 55
mpl.rcParams['font.sans-serif'] = 'Arial'
mpl.rcParams['figure.dpi'] = 500
mpl.rcParams['svg.fonttype'] = 'none'

In [12]:
def set_box_color(bp, color):
    plt.setp(bp['boxes'], color=color)
    plt.setp(bp['whiskers'], color=color)
    plt.setp(bp['caps'], color=color)
    plt.setp(bp['medians'], color=color)


def set_pvalue(pvalue):
    if pvalue < 0.0001:
        starpval = "****"
    elif pvalue < 0.001:
        starpval = "***"
    elif pvalue < 0.01:
        starpval = "**"
    elif pvalue < 0.05:
        starpval = "*"
    else:
        starpval = "ns"
    return starpval

def make_boxplot(df: pd.DataFrame, title:str, color: int, x= 'constr_cyto', y='Efret_ratio', save_fig: None | str = None, ylabel = None, ylim = False, star_values_list = None, y_set= 3):
    plt.figure(figsize=(30, 15))
    
    y1= df.groupby(by=x)[y].apply(list).tolist()
    y1 = [lst for lst in y1 if lst]
    #y2= df.groupby(['constr_cyto'])['directA_ratio'].apply(list).tolist()
    #y2 = [lst for lst in y2 if lst]
    
    len_y = len(y1)
    labels = ["T"+str(item) for item in df[x].unique().tolist()]
    length = len(labels)
    spacing = np.linspace(1, length, length)
    #left_set = spacing -0.2
    #right_set = spacing +0.2
    colors = plt.cm.YlOrRd(np.linspace(0.8, 0.25, len_y)).tolist()

    for i, category_data in enumerate(y1):
        color = colors[i]
        if star_values_list != None:
            star_value = star_values_list[i]
        i = i+1
        x_positions = i + np.random.uniform(-0.12, 0.12, size=len(category_data))  # Add jitter
        plt.scatter(x_positions, category_data, alpha=0.7, s=300, color=color)
        if star_values_list != None:
            plt.text(i, y = y_set, s=star_value, ha='center')
        median_data = np.median(np.array(category_data))
        if median_data < 1.0:
            plt.text(i, 2.85, s="+", ha="center")
    bpl = plt.boxplot(y1, positions=spacing, widths=0.8, showfliers=False)
    #bpr = plt.boxplot(y2, positions=right_set, widths=0.3)
    set_box_color(bpl, 'k')
    #set_box_color(bpr, colors[1])
    
    #plt.plot([], c=colors[color], label=y)
    #plt.plot([], c=colors[1], label='DirectA_ratio')
    #plt.legend()
    for element in ['boxes', 'whiskers', 'caps', 'medians']:
        for line in bpl[element]:
            line.set_linewidth(8)

    if ylabel is not None:
        plt.ylabel(ylabel)
    plt.title(title)
    
    plt.xticks(spacing, labels, fontsize=70)

    plt.yticks([1.0, 2.0])
    plt.axhline(1, c='teal', linestyle='--', linewidth=4)
    if ylim:
        plt.ylim(ylim)
    if save_fig != None:
        plt.savefig(save_fig, format=save_fig.split('.')[1], bbox_inches='tight')
    plt.show()



In [18]:
boxplot_df = pd.read_csv("nuc-cyto_24hr_non.csv")

In [19]:
make_boxplot(boxplot_df, title='', color=1, y='directA_ratio', ylabel='Direct Acceptor Ratio')


In [10]:
plotting_df = pd.read_csv(r"cyto_vs_nu_48hr.csv")

In [16]:
from matplotlib.patches import Rectangle
from pandas.api.types import CategoricalDtype
from matplotlib.colors import to_hex

multicolor=False 

line_colors = [
    to_hex(c) for c in plt.cm.YlOrRd(np.linspace(0.8, 0.25, 12))
]

constructs= ['1','2', '3', '4', '5', '6', '7','8', '9', '10', '11', '12']

plotting_df['construct'] = plotting_df['construct_cyto'].str.split('-').str[1]

cat_type = CategoricalDtype(categories=constructs, ordered=True)
plotting_df['construct'] = plotting_df['construct'].astype(cat_type)

plotting_df = plotting_df.sort_values(['construct'])

fig, ax = plt.subplots(1, 1, sharey='row', figsize=(30, 15))

colors = ['dodgerblue', 'purple']

generator = plotting_df.groupby(['construct'])
for index, tile in enumerate(generator):
    tmp_df = tile[1]

    infected_df, noninfected_df = tmp_df.loc[(tmp_df['infection'] == 'infected')], tmp_df.loc[(tmp_df['infection'] == 'noninfected')]
    
    df_list = [noninfected_df, infected_df]
    
    da_sign_list = []
    
    
    for i in range(2):
        it_df = df_list[i]
        color = colors[i]
        
        
        da_data = it_df['DirectA_ratio']

        da_sign_list.append(da_data)
        

        bp_list_1 = []

        match i:
            case 0:
                x_loc = -0.2
            case 1:
                x_loc = 0.2
        bp1 = ax.boxplot(da_data, positions= [index + x_loc], widths=0.3, showfliers=False)
        

        x_list = []
        if multicolor:
            mcolor = ['k', 'r', 'g', 'b', 'purple', 'yellow']
            for rep in it_df['replicate_cyto'].unique():
                c = mcolor[int(rep)]
                
                rep_data = it_df.loc[it_df['replicate_cyto'] == rep]
                directacceptor_ratio = rep_data['DirectA_ratio']

                data_list = [directacceptor_ratio, efret_ratio]
                x_list = []
                for dataset in data_list:
                    x_list.append([index + x_loc] + np.random.uniform(-0.1, 0.1, size=len(dataset)))
                ax.scatter(x_list[0], directacceptor_ratio, color = c,edgecolor='k', s=500, alpha=0.5)
                

        else:
            color=colors[i]
            data_list = [da_data]
            for dataset in data_list:
                x_list.append([index + x_loc] + np.random.uniform(-0.1, 0.1, size=len(dataset)))
        
            ax.scatter(x_list[0], da_data, color=color, edgecolor='k', s=500, alpha=0.5)
            
        
        list_bp = [bp1]
        for box in list_bp:
            line = line_colors[index]
            set_box_color(box, color='k')
            plt.setp(box['medians'], color=line)
            for element in ['boxes', 'whiskers', 'caps', 'medians']:
                for line in box[element]:
                    line.set_linewidth(8)

    ha= 'center'
    data_variables = [da_sign_list]
    for set_index, data_values in enumerate(data_variables):
        _, pvalue = stats.ttest_ind(data_values[0], data_values[1], equal_var =False)
        star_value = set_pvalue(pvalue) 
        ax.text(index, 2.6, s=star_value, ha=ha, fontsize=50)
        
    ax.axhline(1.0, linestyle='--', color='teal', linewidth=3)

xticks = []
xpos =[]
for x in range(12):
    xpos.append(x)
    xticks.append(f"T{x+1}")
'''for axes in ax[0:6]:
    axes.set_xticklabels([])'''
ax.set_xticks(xpos)
ax.set_xticklabels(xticks, fontsize=60)
ax.set_ylim([0.3, 2.8])
plt.subplots_adjust(hspace=0.05, wspace=0.2)

#plt.savefig(r'cyto_nuclei_boxplot_infection_singleplot.svg', format='svg', bbox_inches='tight')

C:\Users\Owner\AppData\Local\Temp\ipykernel_24552\3929135078.py:24: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  generator = plotting_df.groupby(['construct'])
